### (Optional), another foreground detection function

In [ ]:


# img_ecoli = tiff.imread(path_img_ecoli)
img_ecoli = tiff.imread('/Users/m.wehrens/Data_notbacked/2025_Py-Image-wokshop_Filamentation-example-data/images/pos3crop-p-2-1247.tif')

img_ecoli_inv = np.max(img_ecoli)-img_ecoli
img_ecoli_edges = my_detect_edges(img_ecoli_inv)
mask_ecoli = apply_triangle_n_polish(img_ecoli_inv)
mask_ecoli_edges = apply_triangle_n_polish(img_ecoli_edges)

my_plot_12(mask_ecoli, mask_ecoli_edges)

def detect_foreground_from_edges(mask_ecoli_edges):
    
    # Use the edges picture again, now to create a foreground mask
    
    # dilate then erode, this will fill holes in the microcolony edges part of the image,
    # creating a big blob where the colony was
    mask_ecoli_foreground = sk.morphology.binary_closing(~mask_ecoli_edges, footprint=sk.morphology.disk(20))
    
    # Now select the largest object
    mask_ecoli_fg_labeled = sk.measure.label(mask_ecoli_foreground)
    areas = [p.area for p in sk.measure.regionprops(mask_ecoli_fg_labeled)]
    
    # A mask where the largest object is found
    new_mask = ~((np.argmax(areas)+1) != mask_ecoli_fg_labeled)
    
    # now smooth the shape this mask
    new_mask = sk.morphology.binary_opening(new_mask, footprint=sk.morphology.disk(20))
    
    # Dilate this mask by 5 (like before)
    new_mask = sk.morphology.dilation(new_mask, footprint=sk.morphology.disk(5))
    
    return new_mask

mask_foreground1 = sk.morphology.dilation(mask_ecoli, footprint=sk.morphology.disk(5))
mask_foreground2 = detect_foreground_from_edges(mask_ecoli_edges)
mask_foreground = mask_foreground1&mask_foreground2
my_plot_12(mask_foreground1, mask_foreground2)

mask_seeds_labeled = produce_seeds(mask_ecoli_edges, mask_foreground)
my_plot_12(mask_foreground, mask_seeds_labeled)

mask_ecoli_watershed = sk.segmentation.watershed(~mask_ecoli_edges, markers=mask_seeds_labeled, mask=mask_ecoli_edges)
my_plot_1(mask_ecoli_watershed)  

mask_ecoli_watershed_final = mask_ecoli_watershed.copy()
mask_ecoli_watershed_final[~mask_foreground] = 0
my_plot_1(mask_ecoli_watershed_final) 
